# Pattern Research — Elliott Wave Trading Agent

This notebook lets you interactively explore and research trading patterns
before deciding whether to automate them.

## What you can do here:
1. Visualize Elliott Wave detection on any asset
2. See RSI divergences overlaid on price charts
3. Research new patterns using AI analysis
4. Assess pattern viability before building automated strategies

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from data.yfinance_provider import YFinanceProvider
from patterns.elliott_wave import ElliottWaveDetector
from patterns.rsi import RSIAnalyzer, compute_rsi
from signals.wave3_rsi import Wave3RSISignal

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Load Data & Detect Patterns

In [ ]:
# Configuration — change these to explore different assets
SYMBOL = 'SPY'
TIMEFRAME = '1d'
START = '2022-01-01'
END = '2024-01-01'

provider = YFinanceProvider()
df = provider.fetch_ohlcv(SYMBOL, timeframe=TIMEFRAME, start=START, end=END)
print(f'Loaded {len(df)} bars for {SYMBOL} ({TIMEFRAME})')
df.tail()

In [ ]:
# Detect Elliott Wave structures
wave_detector = ElliottWaveDetector(zigzag_pct=5.0)
wave_matches = wave_detector.detect(df)
structures = wave_detector.find_structures(df)

print(f'Found {len(wave_matches)} wave patterns')
print(f'Found {len(structures)} wave structures')
for s in structures:
    print(f'  {s.direction} | Waves completed: {s.completed_waves} | '
          f'Confidence: {s.confidence:.2f} | '
          f'W2 retrace: {s.wave2_retracement:.1%}')

In [ ]:
# Detect RSI patterns
rsi_analyzer = RSIAnalyzer(period=14)
rsi = rsi_analyzer.compute(df)
rsi_matches = rsi_analyzer.detect(df)

print(f'Found {len(rsi_matches)} RSI patterns')
for m in rsi_matches[-10:]:
    print(f'  {m}')

## 2. Visualize Patterns on Price Chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), height_ratios=[3, 1],
                                sharex=True, gridspec_kw={'hspace': 0.05})

# Price chart
ax1.plot(df.index, df['close'], color='#2196F3', linewidth=1, label='Close')
ax1.fill_between(df.index, df['low'], df['high'], alpha=0.1, color='#2196F3')

# Plot wave structures
colors = {'bullish': '#4CAF50', 'bearish': '#F44336'}
for s in structures:
    pts = s.points
    dates = [df.index[p.index] for p in pts if p.index < len(df)]
    prices = [p.price for p in pts if p.index < len(df)]
    color = colors[s.direction]
    ax1.plot(dates, prices, 'o-', color=color, markersize=8, linewidth=2,
             alpha=0.8, label=f'Wave ({s.direction}, conf={s.confidence:.2f})')
    
    # Label wave points
    labels = ['W0', 'W1', 'W2', 'W3', 'W4', 'W5']
    for j, (d, p) in enumerate(zip(dates, prices)):
        if j < len(labels):
            ax1.annotate(labels[j], (d, p), textcoords='offset points',
                        xytext=(0, 10), fontsize=9, fontweight='bold',
                        ha='center', color=color)
    
    # Draw Fibonacci extension targets
    if s.completed_waves >= 2:
        for mult in [1.618, 2.618]:
            target = s.wave3_target(mult)
            if target:
                ax1.axhline(y=target, color=color, linestyle='--', alpha=0.3,
                           linewidth=0.8)
                ax1.text(df.index[-1], target, f'{mult}x=${target:.0f}',
                        fontsize=8, color=color, alpha=0.6)

ax1.set_title(f'{SYMBOL} — Elliott Wave Detection', fontsize=14)
ax1.set_ylabel('Price')
ax1.legend(loc='upper left', fontsize=8)

# RSI subplot
ax2.plot(df.index, rsi, color='#9C27B0', linewidth=1)
ax2.axhline(y=70, color='red', linestyle='--', alpha=0.5)
ax2.axhline(y=30, color='green', linestyle='--', alpha=0.5)
ax2.axhline(y=50, color='gray', linestyle=':', alpha=0.3)
ax2.fill_between(df.index, 30, rsi.clip(upper=30), alpha=0.2, color='green')
ax2.fill_between(df.index, 70, rsi.clip(lower=70), alpha=0.2, color='red')
ax2.set_ylabel('RSI(14)')
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 3. Generate Wave 3 + RSI Signals

In [ ]:
signal_gen = Wave3RSISignal()
signals = signal_gen.generate(df, SYMBOL)

print(f'Generated {len(signals)} trade signals\n')
for sig in signals:
    print(sig)
    print(f'  Risk/Reward: {sig.reward_risk_ratio:.2f}:1')
    print(f'  Targets: {sig.targets}')
    print()

## 4. Research a New Pattern (AI-Powered)

Use this section to research any pattern before building a detector for it.
Requires an API key set in `.env`.

In [ ]:
# Uncomment and modify to research a different pattern:

# from agents.research_agent import ResearchAgent
# researcher = ResearchAgent()
# 
# result = researcher.research_pattern("""
# Double Bottom pattern with volume confirmation.
# Two roughly equal lows with a moderate peak between them.
# Volume should increase on the second bottom and breakout.
# """)
# print(result)